# 08 · Break it

Chapter 08 takes the finished toy chain of chapters 01–07 and returns three variants of it, each with one thing broken on purpose, scored with the same tools as chapter 07's scoreboard.

In [1]:
import contextlib
import io
import json
import logging
from pathlib import Path

import pandas as pd

from pysuto.tutorial import toy, toy_ext

RUNS = Path("runs")
pd.set_option("display.float_format", "{:.3f}".format)
pd.set_option("display.width", 200)
logging.disable(logging.INFO)


def quiet():
    return contextlib.redirect_stdout(io.StringIO())

Chapters 00–07 showed the pipeline on its good days. This chapter shows three bad ones. Each experiment changes one thing, re-runs Stages 2–5 (or 3–4), and compares the result with the base chain. The rule of the project that falls out of each is named at the end of its section; none of them is a surprise once you have seen it break.

Two housekeeping facts. The experiments live in `pysuto.tutorial.toy_ext`, not in `toy`, and they run under their own registry labels (`toy_contradiction_…`, `toy_phantom`, `toy_tolerance`). Your base chain from chapters 01–07 is therefore still "the latest run" of the toy case, and you can run this chapter twice. The toy also departs from copper in one way that matters here: it has no `ROW`, so a phantom leg has only three regions to hide in.

In [2]:
base = toy_ext.base_ids(RUNS)
print(json.dumps(base, indent=1))

{
 "imp": "imp_20261123_151853_1994e51a",
 "const": "const_20261123_151905_b9419350",
 "opt": "opt_20261123_151908_06e888ed",
 "uncprop": "uncprop_20261123_151913_714e8d7c"
}


## Experiment 1 · σ cannot repair a contradiction

Chapter 07 ended on the planted leg: Montara's concentrate to Portua in 2003, booked a factor 1000 too large in the Customs feed. Chapters 01–07 ran without scale totals, so the contradiction showed up as a z of about −4.8 on the leg itself.

Copper's Stage 3 also carries **scale totals**: soft rows that say "the sum of a node's supply edges equals the sum of their prior" (ADR 0006). They are anchored on the prior, at a cv of 1 % in copper. In a world where one prior is wrong, such a row restates the error with great confidence. `toy_ext.break_contradiction` switches them on at 1 %, then does what a tempted analyst does: it makes the offended row's σ larger, step by step (cv of 1 %, 10 %, 100 % and 1000 %), and looks at the planted edge each time.

The row is the use total of Montara's concentrate flow in 2003. Each variant is a full Stage 3 → 5 chain, so this cell takes a few seconds.

In [3]:
with quiet():
    exp1 = toy_ext.break_contradiction(RUNS)
t1 = exp1["table"]
t1[["row_cv", "planted_row_z", "planted_a_star", "planted_truth", "coverage", "median_rel_err"]]

,row_cv,planted_row_z,planted_a_star,planted_truth,coverage,median_rel_err
variant,,,,,,
base,NaN,NaN,23.628,22.729,0.943,0.031
cv=0.01,0.010,-58.593,23.678,22.729,0.664,0.035
cv=0.1,0.100,-9.618,23.637,22.729,0.664,0.035
cv=1,1.000,-0.997,23.636,22.729,0.664,0.035
cv=10,10.000,-0.100,23.636,22.729,0.664,0.035
fixed,0.010,0.221,23.462,22.729,0.664,0.035


Read the columns one at a time. `planted_row_z` is the z of the contradicted row: the price the QP pays for it, in σ. `planted_a_star` is the solved planted edge, in `kt` (Gross), against its `planted_truth`. `coverage` is the share of edges whose 95 % posterior interval holds the truth (Stage 5, as in chapter 07). The `base` row is the chain without scale totals; `fixed` is the last row, explained below.

In [4]:
z_tight, z_loose = t1.loc["cv=0.01", "planted_row_z"], t1.loc["cv=10", "planted_row_z"]
a_tight, a_loose = t1.loc["cv=0.01", "planted_a_star"], t1.loc["cv=10", "planted_a_star"]
print(f"row z: {z_tight:.1f} at cv 1 % → {z_loose:.2f} at cv 1000 % (÷{z_tight / z_loose:.0f})")
print(f"planted a*: {a_tight:.2f} → {a_loose:.2f} kt (Gross) against a truth of {t1.loc['base', 'planted_truth']:.2f} kt (Gross)")

row z: -58.6 at cv 1 % → -0.10 at cv 1000 % (÷587)
planted a*: 23.68 → 23.64 kt (Gross) against a truth of 22.73 kt (Gross)


Two things happened, and they are not the same thing. The residual of the row went from a number nobody could ignore to a number nobody would look at, about 590 times smaller (a σ 1000 times larger, so z = residual / σ falls by about that; the rest is the residual itself moving a little). The planted edge, meanwhile, hardly moved: `a*` sits near 23.6 kt (Gross) whichever σ we give the row, against a truth of about 22.7 kt (Gross).

That is the surprise of this experiment: the leg was never wrong because of the scale total. It was repaired already, by the hard mass balances at both ends of the leg and by the other sources (chapter 07's planted-edge table: 23,824 kt (Gross) in the feed, about 23.6 kt (Gross) after reconciliation). The scale total only restated the error and booked it as a residual. Loosening its σ moved nothing in the mass; it **hid the symptom**.

The anchors themselves are not free either. Look at `coverage` and at the largest soft residual of each variant.

In [5]:
t1[["coverage", "top_row_z", "top_row"]].loc[["base", "cv=0.01", "cv=10"]]

,coverage,top_row_z,top_row
variant,,,
base,0.943,0.617,ratio_concoresupplyisla_Ore.Isla-Milling.Isla_...
cv=0.01,0.664,58.593,sum_usetotout_Concentrate_Montara_Content_2003
cv=10,0.664,11.510,sum_usetotout_Concentrate_Isla_Content_2002


Whatever the planted row's σ, coverage drops from about 0.94 (base) to about 0.66 once the anchors are in, and another anchor (Isla's concentrate flow in 2002) takes over as the largest residual. The anchors restate the prior of every node at 1 %, and the toy's prior is imputed in places (chapter 02): an imputed number held at 1 % is a claim the data do not support. This is the reason chapters 00–07 run **without** scale totals, and the reason copper floors the σ of its feed-backed totals only (ADR 0025) and never globally.

Now the repair that works: rebuild the Customs feed *without* the planted error and re-run Stages 2–5 with the same anchors at 1 %. That is the `fixed` row.

In [6]:
fixed = t1.loc["fixed"]
print(f"fixed input: row z {fixed['planted_row_z']:+.2f} (was {z_tight:.1f}); planted a* {fixed['planted_a_star']:.2f} kt (Gross)")
print(f"the planted edge moved {abs(fixed['planted_a_star'] - a_tight):.2f} kt (Gross) between the contradicted and the fixed chain")

fixed input: row z +0.22 (was -58.6); planted a* 23.46 kt (Gross)
the planted edge moved 0.22 kt (Gross) between the contradicted and the fixed chain


The contradiction is gone: the row's z is now about +0.2, an ordinary number, with the σ left at 1 %. And the planted edge moved by less than a kt (Gross), because it was never the mass that was wrong. What was wrong was the **input**, and only an edit of the input removes a contradiction from the QP instead of repricing it.

This is ADR 0019 and ADR 0035 in one experiment. In copper, loosening contradicted nodes repriced the disagreement while 0.46 % of the mass moved and the collapses it targeted stayed (ADR 0019); the defects that mattered were fixed in the parse (ADR 0030, 0032, 0033, 0035, 0036). **σ cannot repair a data contradiction: fix the input, never loosen σ or drop the row.**

<details><summary>Going deeper: z is the price of a residual, not the damage it does</summary>

A soft row with residual $r_k = (G a)_k - c_k$ and σ $\sigma_k$ adds $(r_k / \sigma_k)^2$ to the objective of Eq. (2). Its z is $r_k/\sigma_k$. Make $\sigma_k$ a thousand times larger and the term of the objective falls by a factor of a million, the QP stops paying attention to the row, and z falls by a thousand: the residual $r_k$ is whatever the other rows leave it. Nothing in this chain makes $r_k$ small, and nothing could, because the information that the leg is wrong is not in the row, it is in the balances and the other feeds.

So a falling z after a σ loosening has two readings: the contradiction is resolved, or the row has stopped listening. Only the second is available here. The check that tells them apart is the one the table above makes: did the *mass* move? In this experiment `a*` of the planted edge changes by a few hundredths of a kt (Gross) across a thousandfold σ. The same check on copper's loosened nodes, 0.46 % of the mass, gave the same answer (ADR 0019).
</details>

**Exercise 8.1** — From the table, work out by what factor z fell between the cv of 1 % and the cv of 1000 % rows, and say whether the thousandfold σ alone explains it. Cover the next cell and try it first.

In [7]:
factor = t1.loc["cv=0.01", "planted_row_z"] / t1.loc["cv=10", "planted_row_z"]
print(toy_ext.check_8_1(t1, factor))

✓ 8.1  z fell 587× while the row's σ grew 1000×: the residual itself grew 1.70×


The factor is smaller than 1000 because the residual is not fixed: with a larger σ the row loosens its grip and the residual grows a little. The row is not free, it is cheaper.

## Experiment 2 · The model refuses what the data cannot support

In copper, BACI books 838 kt CC of Zambian anode exports in 2022 against a smelter output of 727 kt CC (ADR 0021): a trade record that no producer could have supplied. The model refuses it by capacity, and keeps only about 2 % of the leg (retention 0.02, ADR 0026). The toy has the same story at a size you can hold: `toy_ext.break_phantom` adds one more row to the Customs feed, a leg from Isla's concentrate to Portua's smelter in 2003 of **three times Isla's whole concentrate output** that year, and re-runs Stages 2–4.

In [8]:
with quiet():
    exp2 = toy_ext.break_phantom(RUNS)
exp2["table"].T

,0
leg,Concentrate[Isla] → Smelting[Portua] 2003 Gross
output,10.290
truth,1.543
prior,32.862
a_star,1.889
posterior_over_prior,0.057
z,-4.509


`output` is Isla's concentrate output in 2003, `truth` the real leg, `prior` what Stage 2 passed to the QP once the phantom was aggregated in (the phantom and the real leg are one edge, so they were combined by Eq. (1) and inverse variance), and `a_star` the reconciled value. All in `kt` (Gross).

In [9]:
row = exp2["table"].iloc[0]
print(f"prior {row['prior']:.1f} kt (Gross) against an exporter that makes {row['output']:.1f} kt (Gross)")
print(f"a* {row['a_star']:.2f} kt (Gross): a posterior of {row['posterior_over_prior']:.3f} of the prior; truth {row['truth']:.2f} kt (Gross)")

prior 32.9 kt (Gross) against an exporter that makes 10.3 kt (Gross)
a* 1.89 kt (Gross): a posterior of 0.057 of the prior; truth 1.54 kt (Gross)


The prior is about three times what Isla could ship, and `a*` is about 6 % of the prior, close to the truth of 1.5 kt (Gross). Nothing in the code says "refuse trade larger than output", and the toy has no scale totals to refuse it against (those are copper's, and appear in experiment 1 only). The refusal falls out of the hard mass balances, with the soft ratio rows of Isla's milling chain beside them: the concentrate Isla's mill produces is the most that can leave its Concentrate flow node, and a leg larger than that has to be paid for by moving it. The QP pays, because the alternative is to break a hard row.

How does the pipeline report that it did so? The same way as in chapter 07: as a ranked soft residual, and as a retention far below 1.

In [10]:
board = toy.scoreboard(RUNS, base)
soft2 = toy.soft_residuals(RUNS, exp2["runs"]["opt"]).head(3)
soft2.assign(edge=board.edge_labels(soft2["edge_id"].astype(int)))[["edge", "z"]]

,edge,z
0,Concentrate[Montara] → Smelting[Portua] 2003 G...,-4.779
1,Concentrate[Isla] → Smelting[Portua] 2003 Gross,-4.509
2,Concentrate[Montara] → Smelting[Portua] 2003 C...,-4.031


The phantom leg is second, at a z of about −4.5, just behind the planted Customs leg of chapter 07 that this chain still carries (z about −4.8), and that leg's content twin is third. That is the honest limit of the experiment: the retention of 0.06 says the model deleted most of the leg, and the residual says it was a struggle, but neither says *why*. A small Isla output and a mistaken feed give the same picture; in copper the Zambian case needed a human to read the smelter's output beside the trade record (ADR 0021). The model refuses by capacity; it does not diagnose.

The project rule: **retention below 1 is a question, not a verdict** (GLOSSARY, "Retention"). In copper, ADR 0029 measured that 49 % of the mine-chain deletion was forced by missing export mass: data that contradict the balances, not a modelling choice.

<details><summary>Going deeper: why the leg is not simply zeroed</summary>

$a^*$ is 1.9 kt (Gross) and the truth 1.5 kt (Gross): the posterior is not zero and not exactly the truth. Isla's real concentrate leg to Portua is the same edge as the phantom, and the soft prior still pulls $a^*$ towards its (wrong) value at weight $\sigma_a^{-2}$. The balances bound the sum of Isla's concentrate legs; within that bound the prior decides the split. What survives of the phantom is that pull, a few tenths of a kt (Gross); it is not a second mechanism.
</details>

## Experiment 3 · The termination rule matters

The last experiment leaves the model alone and changes how long the solver runs. Clarabel stops when the **relative duality gap** is below a tolerance; ADR 0039 chose 1e-6 for copper after production runs had stopped at 1e-3. `toy_ext.break_tolerance` solves one year (2003) twice, at a gap of 1e-3 and 1e-6, and compares the two period slices edge by edge, in units of each edge's prior σ.

First on the toy as it is, chapter 03's QP with equality rows only.

In [11]:
with quiet():
    exp3a = toy_ext.break_tolerance(RUNS, bands=False)
s3a = exp3a["summary"]
print(f"iterations at 1e-3 and 1e-6: {s3a['iterations']}; largest difference between the two a*: {s3a['max_abs_diff']} kt (Gross or Content)")

iterations at 1e-3 and 1e-6: (0, 0); largest difference between the two a*: 0.0 kt (Gross or Content)


Zero iterations, and the two solutions are the same to the last digit. This is not a bug and not luck. The toy's QP has no inequality rows (chapter 03's "no bands here"), so Clarabel does not iterate: it solves one linear system, the KKT conditions, directly, and there is no gap to stop on. The tolerance has nothing to bite on.

Copper's QP has an inequality cone, the non-negativity row of every flow (ADR 0039). `bands=True` adds exactly that and nothing else of copper's bands to the toy: `FlowBoundsBuilder`, the non-negativity lower bounds of ADR 0039, as another const run under its own label.

In [12]:
with quiet():
    exp3 = toy_ext.break_tolerance(RUNS, bands=True)
s3 = exp3["summary"]
print(f"iterations at 1e-3 and 1e-6: {s3['iterations']}; objective {s3['objective'][0]:.3f} vs {s3['objective'][1]:.3f}")
print(exp3["table"].to_string(float_format="{:.3g}".format))

iterations at 1e-3 and 1e-6: (12, 16); objective -8611.149 vs -8611.969
                                           edge  a_loose  a_tight     diff  sigma_prior  diff_over_sigma
0      Mining[Portua] → WasteRock[Portua] Gross     12.1    0.434     11.7         43.2            0.271
1  Mining[Montara] → WasteRock[Montara] Content  2.2e-07 7.13e-09 2.13e-07        1e-06            0.213
2    Mining[Portua] → WasteRock[Portua] Content  2.2e-07 7.13e-09 2.13e-07        1e-06            0.213
3        Mining[Isla] → WasteRock[Isla] Content  2.2e-07 7.13e-09 2.13e-07        1e-06            0.213
4  Concentrate[Portua] → Smelting[Isla] Content   0.0403    0.024   0.0163       0.0846            0.193
5        Deposit[Portua] → Mining[Portua] Gross     6.07    0.217     5.85         30.8             0.19


Now the two runs differ: the loose solve stops after fewer iterations, at another point, with a slightly higher objective. The table lists the six edges that moved most, with `diff_over_sigma` their difference measured in prior σ.

In [13]:
print(f"largest difference: {s3['max_abs_diff']:.1f} kt (Gross) = {s3['max_diff_over_sigma']:.2f} σ_prior; edges differing by more than their σ: {s3['n_over_sigma']}")

largest difference: 11.7 kt (Gross) = 0.27 σ_prior; edges differing by more than their σ: 0


On the toy the answer is gentle: the largest difference is about a quarter of a σ, on Portua's `Mining → WasteRock` flow, a free slack that no feed observes and that carries no `Content`. **No edge moves by more than its σ**, so a reader of the toy's results would not notice the tolerance. The honest reading is that the toy is too small and too tightly weighted to show the damage. What it does show is the mechanism: with inequality rows the iterate where the gap test passes is not the optimum, and the edges with the loosest σ are the ones that pay.

Copper is where it bit. ADR 0039 re-solved 2015 of one constraint run (W.3) at a gap of 1e-8 instead of the production 1e-3: the objective fell by 9,052 (8 % of the misfit), and Chile's concentrate exports moved from 1,658 to 2,452 kt CC against a prior of 2,522. That is a retention of 0.66 against 0.97, on the same QP (REP `…roundW1-lower-bounds`, runs `opt_20260929_121937_e5f000b5` and `opt_20260930_092111_e3e0677d`; manuscript Table 2). Copper's driver has run each year to its numerical floor since.

The project rule: **a retention is only as good as its termination.** A comparison across a solver tolerance mixes the model's effect with the solver's (ADR 0039, Consequences).

<details><summary>Going deeper: why a relative gap says little here</summary>

Solved for the flows, Clarabel minimises $J - \sum (a_0/\sigma_a)^2$: a constant that is large against the misfit (in copper 2015, the constant is 1.17e6 and the misfit 1.13e5). A relative gap of $10^{-3}$ of that cost admits about 1,000 objective units of error. ADR 0039's later amendment solves for the defects $d = a - a_0$ instead, so the cost is the misfit itself and the gap certifies what it says. The toy still solves for the flows (`QP_VARIABLES = "flows"`), so the effect above is reproducible here.
</details>

## What the three have in common

| Experiment | What was broken | What the pipeline did | Project rule | ADR |
| --- | --- | --- | --- | --- |
| 1 | σ of the row an erroneous prior contradicts | z fell, mass did not move | σ cannot repair a data contradiction; fix the input | 0019, 0035 |
| 2 | a trade leg larger than its exporter's output | the hard mass balances deleted most of it; the residual flagged it | the model refuses what the data cannot support; retention is a question | 0021, 0026, 0029 |
| 3 | the solver's termination | the solver stopped at another point; small σ moved little | report and test the termination | 0039 |

In every case the information that decided the outcome was already in the data. The tutorial's toy world was balanced by construction, which is why each break is legible. Copper's world is not; chapter 09 maps one onto the other.

## Where this run came from

The experiments write their own runs beside the base chain. The sidecar of the phantom experiment's optimiser run shows its chain:

In [14]:
sidecar = json.loads((RUNS / "proc" / "opt" / exp2["runs"]["opt"] / "metadata.json").read_text())
print(sidecar["label"])
print(json.dumps(sidecar["parents"], indent=2))

toy_phantom
{
  "inputs": {},
  "upstream": {
    "const_run_id": "const_20261123_151935_9de531e7"
  }
}


The `const_run_id` points to the Stage 3 run of the variant, built on the `imp_run_id` that aggregated the planted Customs feed. Chapter 09 leaves the toy for copper.